# Figure 1. Modular dual-atom catalyst design space

This notebook downloads the clean catalyst database, selects one representative dual-atom catalyst, and draws the structural variables used to construct the catalyst library: metal 1, metal 2, bridge, and diamine.

The database structure is used directly. Atom connectivity is analysed only to locate the modules for the callout arrows; no atoms or coordinates are changed.

## Release v30
This self-contained release downloads the exact GitHub v1 database, verifies its SHA-256 hash, flips the molecular drawing horizontally, and preserves normal text direction.


In [ ]:
###############################
##-Install required packages-##
###############################

!pip install -q ase matplotlib pillow
!apt-get -qq update
!apt-get -qq install -y povray

In [ ]:
#########################
##-Importing libraries-##
#########################

from contextlib import contextmanager
from pathlib import Path
from tempfile import TemporaryDirectory
import hashlib
import os
import shutil
import urllib.request

import matplotlib.pyplot as plt
from matplotlib.colors import to_rgb
from matplotlib.path import Path as MplPath
import numpy as np
from PIL import Image
from ase.data import covalent_radii
from ase.data.colors import jmol_colors
from ase.db import connect
from ase.io import write
from ase.io.pov import get_bondpairs
from ase.neighborlist import NeighborList, natural_cutoffs

%matplotlib inline
IN_COLAB = Path('/content').is_dir()

plt.rcParams.update(
    {
        'figure.dpi':     120,
        'savefig.dpi':    300,
        'font.family':    'sans-serif',
        'font.size':      10,
        'axes.titlesize': 11,
    }
)

In [ ]:
####################################
##-Load the publication database-##
####################################

DATABASE_NAME   = 'DASC_Bare_Catalysts_RPBE_D4.db'
DATABASE_URL    = (
    'https://raw.githubusercontent.com/doublelayer/'
    'EActa_2026_AI-acceleration/v1/' + DATABASE_NAME
)
DATABASE_SHA256 = '8d185dcf3464860a4ffecf69e67ea4c85c7673cd1468e8c94be119ad9ee5fa53'

if IN_COLAB:
    DATABASE_DIR = Path('/content')
else:
    DATABASE_DIR = Path.cwd() / 'source_data'

def file_sha256(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b''):
            digest.update(chunk)
    return digest.hexdigest()


def is_valid_database(path, expected_sha256):
    if not path.is_file() or path.stat().st_size < 16:
        return False
    with path.open('rb') as handle:
        if handle.read(16) != b'SQLite format 3\x00':
            return False
    return file_sha256(path) == expected_sha256


def obtain_database(filename, url, expected_sha256, directory):
    directory.mkdir(parents=True, exist_ok=True)
    path = directory / filename
    if is_valid_database(path, expected_sha256):
        status = 'reused verified local database'
    else:
        partial = path.with_suffix(path.suffix + '.part')
        partial.unlink(missing_ok=True)
        request = urllib.request.Request(
            url, headers={'User-Agent': 'EActa-2026-database-loader'}
        )
        try:
            with urllib.request.urlopen(request) as response, partial.open('wb') as handle:
                shutil.copyfileobj(response, handle)
            if not is_valid_database(partial, expected_sha256):
                raise RuntimeError(f'Database validation failed: {filename}')
            partial.replace(path)
        finally:
            partial.unlink(missing_ok=True)
        status = 'downloaded and verified from GitHub'
    return path, status, file_sha256(path)

DATABASE_PATH, database_status, actual_hash = obtain_database(
    DATABASE_NAME, DATABASE_URL, DATABASE_SHA256, DATABASE_DIR
)
database = connect(DATABASE_PATH)
print(f'Database file: {DATABASE_NAME}')
print(f'Database source: {DATABASE_URL}')
print(f'Database status: {database_status}')
print(f'Database SHA-256: {actual_hash}')
print(f'Catalyst structures: {database.count()}')
print(f'POV-Ray available: {shutil.which("povray") is not None}')


## Select the representative catalyst

The publication figure uses a Co–Co catalyst with acid fragment A2, base fragment B1, and an O bridge. The query below makes that choice explicit.

In [ ]:
##################################
##-Representative catalyst query-##
##################################

REPRESENTATIVE = {
    'metal1': 'Co',
    'metal2': 'Co',
    'acid':   'A2',
    'base':   'B1',
    'bridge': 'OO',
}

matches = list(database.select(**REPRESENTATIVE))
if not matches:
    raise ValueError(f'No catalyst matches {REPRESENTATIVE}')

representative = matches[0]
atoms          = representative.toatoms()
rows           = list(database.select())

print(f'Formula:  {representative.formula}')

## Locate the structural modules

The two metal atoms are checked against the database metadata. The two bridge atoms are identified as the heteroatoms simultaneously nearest to both metals. Removing the metal–bridge core separates the remaining framework into connected fragments, allowing the acid-derived and base-derived regions to be located.

In [ ]:
####################################
##-Identify the structural modules-##
####################################

def infer_assembly_groups(row):
    structure = row.toatoms()

    metal_indices = [0, 1]
    expected       = [row.metal1, row.metal2]
    observed       = [structure[index].symbol for index in metal_indices]

    if observed != expected:
        unused        = set(range(len(structure)))
        metal_indices = []
        for symbol in expected:
            candidates = [
                index for index in unused
                if structure[index].symbol == symbol
            ]
            if not candidates:
                raise ValueError(f'Cannot locate metal {symbol} in {row.name}')
            chosen = candidates[0]
            metal_indices.append(chosen)
            unused.remove(chosen)

    metal1_index, metal2_index = metal_indices
    heteroatom_indices = [
        index for index, atom in enumerate(structure)
        if index not in metal_indices and atom.symbol in {'N', 'O', 'S'}
    ]
    bridge_indices = sorted(
        heteroatom_indices,
        key=lambda index: max(
            structure.get_distance(index, metal1_index),
            structure.get_distance(index, metal2_index),
        ),
    )[:2]

    remaining_nitrogen_indices = [
        index for index in heteroatom_indices
        if structure[index].symbol == 'N' and index not in bridge_indices
    ]
    coordinating_nitrogen_indices = sorted(
        remaining_nitrogen_indices,
        key=lambda index: min(
            structure.get_distance(index, metal1_index),
            structure.get_distance(index, metal2_index),
        ),
    )[:4]

    core = set(metal_indices + bridge_indices + coordinating_nitrogen_indices)
    neighbours = NeighborList(
        natural_cutoffs(structure, mult=1.0),
        self_interaction=False,
        bothways=True,
    )
    neighbours.update(structure)

    unvisited  = set(range(len(structure))) - core
    components = []
    while unvisited:
        start     = unvisited.pop()
        component = {start}
        queue     = [start]
        while queue:
            atom_index = queue.pop()
            for neighbour_index in neighbours.get_neighbors(atom_index)[0]:
                neighbour_index = int(neighbour_index)
                if neighbour_index in unvisited:
                    unvisited.remove(neighbour_index)
                    component.add(neighbour_index)
                    queue.append(neighbour_index)
        components.append(component)

    acid_indices   = set()
    base_indices   = set()
    bridge_indices = set(bridge_indices)
    for component in components:
        adjacent_indices = {
            int(neighbour_index)
            for atom_index in component
            for neighbour_index in neighbours.get_neighbors(atom_index)[0]
        }
        if adjacent_indices & bridge_indices:
            acid_indices.update(component)
        else:
            base_indices.update(component)

    base_indices.update(coordinating_nitrogen_indices)

    return {
        'metal1': {metal1_index},
        'metal2': {metal2_index},
        'bridge': bridge_indices,
        'acid':   acid_indices,
        'base':   base_indices,
    }


groups = infer_assembly_groups(representative)
assigned_indices = [
    index
    for module_indices in groups.values()
    for index in module_indices
]
if len(assigned_indices) != len(set(assigned_indices)):
    raise ValueError('The structural modules overlap.')
if set(assigned_indices) != set(range(len(atoms))):
    raise ValueError('The structural modules do not cover every atom.')

bridge_symbols = ''.join(sorted(atoms[index].symbol for index in groups['bridge']))
expected_bridge_symbols = ''.join(sorted(representative.bridge.replace(',', '')))
if bridge_symbols != expected_bridge_symbols:
    raise ValueError(
        f'Bridge mismatch: structure gives {bridge_symbols}, '
        f'whereas the metadata gives {expected_bridge_symbols}.'
    )

print({module: len(indices) for module, indices in groups.items()})

In [ ]:
#############################################
##-Render the catalyst in conventional colors-##
#############################################
import re

BOND_RADIUS = 0.10          # Å, single bonds
DOUBLE_BOND_RADIUS = 0.06   # Å, each stick of a double bond
DOUBLE_BOND_OFFSET = 0.11   # Å, half the spacing between the two sticks
IMINE_MAX_LENGTH = 1.34     # Å, C=N imine ~1.29, C-N single ~1.41

render_metal_indices = groups['metal1'] | groups['metal2']

@contextmanager
def working_directory(path):
    previous_directory = Path.cwd()
    os.chdir(path)
    try:
        yield
    finally:
        os.chdir(previous_directory)

def bond_list(structure):
    """Bonds as in the ASE render, without metal-metal pairs."""
    return sorted({
        tuple(sorted((int(a), int(b))))
        for a, b, _ in get_bondpairs(structure, radius=1.0)
        if a != b and not (a in render_metal_indices and b in render_metal_indices)
    })

def kekule_double_bonds(structure, bonds):
    """Imine C=N from bond length, alternating double bonds in carbon six-rings."""
    double = {
        (a, b) for a, b in bonds
        if {structure[a].symbol, structure[b].symbol} == {'C', 'N'}
        and structure.get_distance(a, b) < IMINE_MAX_LENGTH
    }
    used = {index for pair in double for index in pair}

    adjacency = {}
    for a, b in bonds:
        if structure[a].symbol == 'C' and structure[b].symbol == 'C':
            adjacency.setdefault(a, set()).add(b)
            adjacency.setdefault(b, set()).add(a)

    rings = {}
    def walk(path):
        if len(path) == 6:
            if path[0] in adjacency[path[-1]]:
                rings.setdefault(frozenset(path), tuple(path))
            return
        for neighbour in adjacency[path[-1]]:
            if neighbour not in path:
                walk(path + [neighbour])
    for start in adjacency:
        walk([start])

    for ring in rings.values():
        for first in (0, 1):
            option = [
                tuple(sorted((ring[i], ring[(i + 1) % 6])))
                for i in range(first, 6, 2)
            ]
            if not any(index in used for pair in option for index in pair):
                double.update(option)
                used.update(index for pair in option for index in pair)
                break
        else:
            raise ValueError(f'Could not assign Kekule bonds to ring {ring}.')
    return double

ATOM_LINE = re.compile(r'^atom\(<([^>]*)>.*,\s*(\w+)\)')

def add_bonds_to_pov(pov_path, colors, bonds, double_bonds):
    """Append bond cylinders to the POV-Ray scene, using ASE's own atom coordinates."""
    text = pov_path.read_text()
    positions, finish = [], None
    for line in text.splitlines():
        match = ATOM_LINE.match(line.strip())
        if match:
            positions.append([float(value) for value in match.group(1).split(',')])
            finish = match.group(2)
    if len(positions) != len(colors):
        raise ValueError(
            f'Found {len(positions)} atoms in the POV file, expected {len(colors)}.'
        )
    positions = np.array(positions)
    _, _, axes = np.linalg.svd(positions - positions.mean(axis=0))
    plane_normal = axes[2]

    def cylinder(start, end, radius, colour):
        return (
            f'cylinder {{<{start[0]:.4f}, {start[1]:.4f}, {start[2]:.4f}>, '
            f'<{end[0]:.4f}, {end[1]:.4f}, {end[2]:.4f}>, {radius:.4f} '
            f'texture {{pigment {{rgb <{colour[0]:.3f}, {colour[1]:.3f}, {colour[2]:.3f}>}} '
            f'finish {{{finish}}}}}}}\n'
        )

    lines = []
    for a, b in bonds:
        start, end = positions[a], positions[b]
        middle = (start + end) / 2
        if (a, b) in double_bonds:
            side = np.cross(end - start, plane_normal)
            side = side / np.linalg.norm(side) * DOUBLE_BOND_OFFSET
            for shift in (side, -side):
                lines.append(cylinder(start + shift, middle + shift, DOUBLE_BOND_RADIUS, colors[a]))
                lines.append(cylinder(middle + shift, end + shift, DOUBLE_BOND_RADIUS, colors[b]))
        else:
            lines.append(cylinder(start, middle, BOND_RADIUS, colors[a]))
            lines.append(cylinder(middle, end, BOND_RADIUS, colors[b]))
    pov_path.write_text(text + '\n' + ''.join(lines))

def render_structure(structure, width=1300):
    structure = structure.copy()
    structure.set_pbc(False)
    structure.center(vacuum=3.0)
    colors = jmol_colors[structure.numbers].copy()
    radii = covalent_radii[structure.numbers] * 0.42
    bonds = bond_list(structure)
    double_bonds = kekule_double_bonds(structure, bonds)
    print(f'Bonds: {len(bonds)} ({len(double_bonds)} double)')

    render_directory = Path('/content/inputs') if IN_COLAB else Path.cwd() / 'inputs'
    render_directory.mkdir(parents=True, exist_ok=True)
    with TemporaryDirectory(dir=render_directory) as temporary_directory:
        temporary_directory = Path(temporary_directory)
        png_path = temporary_directory / 'representative.png'
        rendered_with_povray = False
        if shutil.which('povray'):
            try:
                with working_directory(temporary_directory):
                    renderer = write(
                        'representative.pov',
                        structure,
                        format='pov',
                        rotation='0x,0y,0z',
                        colors=colors,
                        radii=radii,
                        show_unit_cell=0,
                        povray_settings={
                            'canvas_width': width,
                            'transparent': True,
                            'camera_type': 'orthographic',
                            'bondatoms': [],   # bonds are added below
                            'area_light': [
                                (2.0, 3.0, 40.0), 'White', 0.7, 0.7, 3, 3
                            ],
                        },
                    )
                    add_bonds_to_pov(
                        temporary_directory / 'representative.pov',
                        colors, bonds, double_bonds,
                    )
                    renderer.render()
                rendered_with_povray = True
            except Exception as error:
                print(f'POV-Ray failed ({error}); using the ASE renderer.')
        if not rendered_with_povray:
            write(
                png_path,
                structure,
                format='png',
                rotation='0x,0y,0z',
                colors=colors,
                radii=radii,
                show_unit_cell=0,
            )
        with Image.open(png_path) as image:
            return np.asarray(image.convert('RGBA')).copy()

structure_image = render_structure(atoms)
structure_image = np.fliplr(structure_image).copy()
print(f'Rendered and horizontally flipped image shape: {structure_image.shape}')

## Draw Figure 1

Each callout occupies its own sector around the catalyst. The arrow endpoints identify distinct structural regions, and the arrows do not intersect. The available elements and diamines are read from the database so that the labels update when the database is replaced.

In [ ]:
###################
##-Draw Figure 1-##
###################

##-Style-##
FIGURE_WIDTH_CM = 8.25
FONT_SIZE = 8
plt.rcParams.update({'font.size': FONT_SIZE})
PREVIEW_DPI = 250              # on-screen only
SAVE_DPI = 600

FIGURE_OUTPUT_DIR = Path('/content/inputs') if IN_COLAB else Path.cwd() / 'inputs'
FIGURE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
existing_figure_versions = []
for output_file in FIGURE_OUTPUT_DIR.glob('Figure_2_modular_design_space_v*.png'):
    try:
        existing_figure_versions.append(int(output_file.stem.rsplit('_v', 1)[1]))
    except ValueError:
        pass
FIGURE_VERSION = max(existing_figure_versions, default=31) + 1
OUTPUT_FILES = [
    FIGURE_OUTPUT_DIR / f'Figure_2_modular_design_space_v{FIGURE_VERSION}.png',
    FIGURE_OUTPUT_DIR / f'Figure_2_modular_design_space_v{FIGURE_VERSION}.svg',
    FIGURE_OUTPUT_DIR / f'Figure_2_modular_design_space_v{FIGURE_VERSION}.pdf',
]

METAL_COLOUR = '#F090A0'       # pink
BRIDGE_COLOUR = '#FF2A2A'      # red
DIAMINE_COLOUR = '#50D050'     # green
DIALDEHYDE_COLOUR = '#E69F00'  # orange

HIGHLIGHT_STRENGTH = 0.55      # 0 = invisible, 1 = full colour
HALO_WIDTH_PX = 14             # highlight width beyond each atom
BOX_CLEARANCE_PX = 3           # minimum gap between labels and atoms/bonds
ROW_SHIFT_PX = 20              # extra offset: top row up, bottom row down
CROP_MARGIN_PX = 10            # white margin above and below the labels

box_style = {'boxstyle': 'round,pad=0.4', 'fc': 'white', 'lw': 0.8}
arrow_style = {
    'arrowstyle': '-|>', 'color': 'black', 'lw': 2.2,
    'mutation_scale': 9, 'shrinkA': 3, 'shrinkB': 0,
}

##-Label contents, read from the database-##
def unique_values(field):
    return sorted({getattr(row, field) for row in rows})

chemical_metal_order = ['Cr', 'Mn', 'Fe', 'Co', 'Ni', 'Cu']
available_metals = set(unique_values('metal1')) | set(unique_values('metal2'))
metals = [metal for metal in chemical_metal_order if metal in available_metals]
bridges = ['N', 'O', 'S']
diamines = ['B1', 'B2']
dialdehydes = ['A2']


##-Map atom coordinates to image pixels, calibrated on the rendered image-##
def atom_pixel_positions(structure, image):
    structure = structure.copy()
    structure.set_pbc(False)
    structure.center(vacuum=3.0)
    radii = covalent_radii[structure.numbers] * 0.42
    xy = structure.positions[:, :2]
    lower = (xy - radii[:, None]).min(axis=0)
    upper = (xy + radii[:, None]).max(axis=0)
    opaque_rows, opaque_cols = np.nonzero(image[..., 3] > 10)
    px0, px1 = opaque_cols.min(), opaque_cols.max() + 1
    py0, py1 = opaque_rows.min(), opaque_rows.max() + 1
    scale_x = (px1 - px0) / (upper[0] - lower[0])
    scale_y = (py1 - py0) / (upper[1] - lower[1])
    if abs(scale_x - scale_y) / scale_x > 0.03:
        raise ValueError('Inconsistent pixel scales; check the render transparency.')
    scale = (scale_x + scale_y) / 2
    pixels = np.column_stack([
        px0 + (xy[:, 0] - lower[0]) * scale,
        py0 + (upper[1] - xy[:, 1]) * scale,
    ])
    return pixels, radii * scale, scale

pixels, radii_px, pixel_scale = atom_pixel_positions(atoms, structure_image)
image_height, image_width = structure_image.shape[:2]
pixels[:, 0] = image_width - 1 - pixels[:, 0]
bond_radius_px = 0.10 * pixel_scale

##-Bonds (same criterion as the render, without metal-metal)-##
metal_indices = groups['metal1'] | groups['metal2']
bond_structure = atoms.copy()
bond_structure.set_pbc(False)
bond_pairs = sorted({
    tuple(sorted((int(a), int(b))))
    for a, b, _ in get_bondpairs(bond_structure, radius=1.0)
    if a != b and not (a in metal_indices and b in metal_indices)
})
bridge_unit = metal_indices | groups['bridge']   # Co-O-Co


##-Six-membered carbon rings, atoms in ring order-##
def six_rings(indices):
    carbons = {index for index in indices if atoms[index].symbol == 'C'}
    adjacency = {index: set() for index in carbons}
    for a, b in bond_pairs:
        if a in carbons and b in carbons:
            adjacency[a].add(b)
            adjacency[b].add(a)
    rings = {}

    def walk(path):
        if len(path) == 6:
            if path[0] in adjacency[path[-1]]:
                rings.setdefault(frozenset(path), tuple(path))
            return
        for neighbour in adjacency[path[-1]]:
            if neighbour not in path:
                walk(path + [neighbour])

    for start in carbons:
        walk([start])
    return list(rings.values())

def ring_centre(ring):
    return pixels[list(ring)].mean(axis=0)

diamine_rings = six_rings(groups['base'])
dialdehyde_rings = six_rings(groups['acid'])
if len(diamine_rings) != 2 or len(dialdehyde_rings) != 2:
    raise ValueError('Expected two diamine rings and two dialdehyde rings.')
left_diamine_ring = min(diamine_rings, key=lambda ring: ring_centre(ring)[0])
right_diamine_ring = max(diamine_rings, key=lambda ring: ring_centre(ring)[0])
top_dialdehyde_ring = min(dialdehyde_rings, key=lambda ring: ring_centre(ring)[1])
bottom_dialdehyde_ring = max(dialdehyde_rings, key=lambda ring: ring_centre(ring)[1])

right_metal = max(metal_indices, key=lambda index: pixels[index, 0])
top_bridge = min(groups['bridge'], key=lambda index: pixels[index, 1])

##-Highlight layer: each region painted once, later regions on top (no mixing)-##
def tint(colour, amount):
    rgb = np.array(to_rgb(colour))
    return 1 - amount * (1 - rgb)

pad = int(np.ceil(radii_px.max() + HALO_WIDTH_PX + 5))
layer = np.ones((image_height + 2 * pad, image_width + 2 * pad, 3))

def fill_polygon(mask, vertices):
    vertices = np.asarray(vertices) + pad
    x0 = max(int(vertices[:, 0].min()), 0)
    x1 = min(int(vertices[:, 0].max()) + 2, mask.shape[1])
    y0 = max(int(vertices[:, 1].min()), 0)
    y1 = min(int(vertices[:, 1].max()) + 2, mask.shape[0])
    yy, xx = np.mgrid[y0:y1, x0:x1]
    inside = MplPath(vertices).contains_points(
        np.column_stack([xx.ravel() + 0.5, yy.ravel() + 0.5]))
    mask[y0:y1, x0:x1] |= inside.reshape(xx.shape)

def fill_disc(mask, centre, radius):
    cx, cy = centre + pad
    x0, x1 = max(int(cx - radius), 0), min(int(cx + radius) + 2, mask.shape[1])
    y0, y1 = max(int(cy - radius), 0), min(int(cy + radius) + 2, mask.shape[0])
    yy, xx = np.mgrid[y0:y1, x0:x1]
    mask[y0:y1, x0:x1] |= (xx + 0.5 - cx)**2 + (yy + 0.5 - cy)**2 <= radius**2

def unit_mask(indices, halo, rings=()):
    mask = np.zeros(layer.shape[:2], dtype=bool)
    for index in indices:
        fill_disc(mask, pixels[index], radii_px[index] + halo)
    for a, b in bond_pairs:
        if a in indices and b in indices:
            start, end = pixels[a], pixels[b]
            direction = end - start
            normal = np.array([-direction[1], direction[0]])
            normal = normal / np.linalg.norm(normal) * (bond_radius_px + halo)
            fill_polygon(mask, [start + normal, end + normal, end - normal, start - normal])
    for ring in rings:
        fill_polygon(mask, pixels[list(ring)])
    return mask

##-Modules, painted in order; the metal goes last so it stays purple-##
for indices, colour, rings in [
    (groups['base'], DIAMINE_COLOUR, diamine_rings),
    (groups['acid'], DIALDEHYDE_COLOUR, dialdehyde_rings),
    (bridge_unit, BRIDGE_COLOUR, ()),
    (metal_indices, METAL_COLOUR, ()),
]:
    layer[unit_mask(indices, HALO_WIDTH_PX, rings)] = tint(colour, HIGHLIGHT_STRENGTH)

##-Figure: fixed width, axes fill the figure, same scale in x and y-##
data_width = image_width + 2 * pad
inches_per_unit = (FIGURE_WIDTH_CM / 2.54) / data_width
extra = 0.5 * image_height                       # temporary room for the labels
y_top, y_bottom = -pad - extra, image_height + pad + extra
figure = plt.figure(
    dpi=PREVIEW_DPI,
    figsize=(FIGURE_WIDTH_CM / 2.54, (y_bottom - y_top) * inches_per_unit),
    facecolor='white',
)
axis = figure.add_axes([0, 0, 1, 1])
axis.imshow(layer, extent=(-pad, image_width + pad, image_height + pad, -pad),
            zorder=0, interpolation='bilinear')
axis.imshow(structure_image, extent=(0, image_width, image_height, 0), zorder=1)
axis.set_xlim(-pad, image_width + pad)
axis.set_ylim(y_bottom, y_top)
axis.axis('off')

##-Obstacles for label placement: atoms and points along bonds-##
obstacle_centres = [pixels[index] for index in range(len(atoms))]
obstacle_radii = [radii_px[index] + BOX_CLEARANCE_PX for index in range(len(atoms))]
for a, b in bond_pairs:
    for fraction in (0.25, 0.5, 0.75):
        obstacle_centres.append(pixels[a] + (pixels[b] - pixels[a]) * fraction)
        obstacle_radii.append(bond_radius_px + BOX_CLEARANCE_PX)
obstacle_centres = np.array(obstacle_centres)
obstacle_radii = np.array(obstacle_radii)

def box_collides(x0, x1, y0, y1):
    nearest_x = np.clip(obstacle_centres[:, 0], x0, x1)
    nearest_y = np.clip(obstacle_centres[:, 1], y0, y1)
    distance_sq = (obstacle_centres[:, 0] - nearest_x)**2 + (obstacle_centres[:, 1] - nearest_y)**2
    return np.any(distance_sq < obstacle_radii**2)

def measure_box(label):
    probe = axis.text(0, 0, label, ha='center', va='center', multialignment='left',
                      fontsize=FONT_SIZE, bbox={**box_style, 'ec': 'black'})
    figure.canvas.draw()
    extent = probe.get_bbox_patch().get_window_extent(figure.canvas.get_renderer())
    (x0, y0), (x1, y1) = axis.transData.inverted().transform(
        [[extent.x0, extent.y0], [extent.x1, extent.y1]])
    probe.remove()
    return abs(x1 - x0), abs(y1 - y0)
def measure_text(label):
    """Width and height of the text alone (no box padding), in data units."""
    probe = axis.text(0, 0, label, ha='center', va='center', multialignment='left',
                      fontsize=FONT_SIZE)
    figure.canvas.draw()
    extent = probe.get_window_extent(figure.canvas.get_renderer())
    (x0, y0), (x1, y1) = axis.transData.inverted().transform(
        [[extent.x0, extent.y0], [extent.x1, extent.y1]])
    probe.remove()
    return abs(x1 - x0), abs(y1 - y0)

def free_y(x0, width, height, start_y, step):
    y = start_y
    for _ in range(5000):
        if not box_collides(x0, x0 + width, y - height / 2, y + height / 2):
            return y
        y += step
    raise ValueError('Could not place a label.')

def draw_callout(label, box_xy, start, tip, border_colour):
    ##-Arrow first, exactly from start to tip-##
    axis.annotate(
        '', xy=tuple(tip), xycoords='data',
        xytext=tuple(start), textcoords='data',
        arrowprops={**arrow_style, 'shrinkA': 0},
        zorder=4, annotation_clip=False,
    )
    ##-Box on top, so the arrow appears to leave its edge-##
    axis.text(
        box_xy[0], box_xy[1], label,
        ha='center', va='center', multialignment='left', fontsize=FONT_SIZE,
        bbox={**box_style, 'ec': border_colour},
        zorder=5, clip_on=False,
    )

##-Ring-centred guide grid for the four label positions-##
DIALDEHYDE_SHIFT_PX = 40       # moves the dialdehyde box down

left_x = ring_centre(left_diamine_ring)[0]
right_x = ring_centre(right_diamine_ring)[0]
top_y = pixels[list(top_dialdehyde_ring), 1].min()
bottom_y = pixels[list(bottom_dialdehyde_ring), 1].max()
callouts = [
    ('top', f'bridge: {", ".join(bridges)}',
     np.array([left_x, top_y]), pixels[top_bridge], BRIDGE_COLOUR),
    ('top', f'metal: {", ".join(metals[:2])},\n{", ".join(metals[2:])}',
     np.array([right_x, top_y]), pixels[right_metal], METAL_COLOUR),
    ('bottom', f'dialdehyde: {", ".join(dialdehydes)}',
     np.array([left_x, bottom_y + DIALDEHYDE_SHIFT_PX]),
     ring_centre(bottom_dialdehyde_ring), DIALDEHYDE_COLOUR),
    ('bottom', f'diamine: {", ".join(diamines)}',
     np.array([right_x, bottom_y]), ring_centre(right_diamine_ring), DIAMINE_COLOUR),
]

##-Arrows run parallel to the nearest bond at their target-##
SNAP_TO = {
    'metal': {right_metal},
    'bridge': {top_bridge},
    'diamine': set(right_diamine_ring),
    'dialdehyde': set(bottom_dialdehyde_ring),
}

def bond_directions(atom_indices):
    """Unit vectors of all bonds touching the given atoms."""
    directions = []
    for a, b in bond_pairs:
        if a in atom_indices or b in atom_indices:
            vector = pixels[b] - pixels[a]
            directions.append(vector / np.linalg.norm(vector))
    return directions

box_edges = []   # (top, bottom) of every drawn box, for the crop
for row, label, box_xy, target, colour in callouts:
    key = label.split(':')[0]
    width, height = measure_box(label)
    box_left, box_right = box_xy[0] - width / 2, box_xy[0] + width / 2
    box_top, box_bottom = box_xy[1] - height / 2, box_xy[1] + height / 2
    box_edges.append((box_top, box_bottom))
    if key == 'dialdehyde':
        ##-Straight from the right edge of the left box to the ring centre-##
        EDGE_FRACTION = 0.25   # 0 = top of the box, 0.5 = middle, 1 = bottom
        start = np.array([box_right, box_top + EDGE_FRACTION * height])
        draw_callout(label, box_xy, start, target, colour)
        continue
        ##-Where the bond-parallel line through the target crosses the box outline-##
    reference = (target - box_xy) / np.linalg.norm(target - box_xy)
    direction = max(bond_directions(SNAP_TO[key]), key=lambda u: abs(u @ reference))
    crossings = []
    if abs(direction[0]) > 1e-6:
        for x in (box_left, box_right):
            y = target[1] + (x - target[0]) * direction[1] / direction[0]
            if box_top <= y <= box_bottom:
                crossings.append(np.array([x, y]))
    if abs(direction[1]) > 1e-6:
        for y in (box_top, box_bottom):
            x = target[0] + (y - target[1]) * direction[0] / direction[1]
            if box_left <= x <= box_right:
                crossings.append(np.array([x, y]))
    tip = target
    if crossings:
        start = min(crossings, key=lambda point: np.linalg.norm(point - target))
    else:
        ##-Line misses the box: start at the nearest corner, keep the bond angle-##
        corners = [np.array([x, y]) for x in (box_left, box_right)
                   for y in (box_top, box_bottom)]
        start = min(corners, key=lambda point: np.linalg.norm(point - target))
        tip = start + ((target - start) @ direction) * direction

    draw_callout(label, box_xy, start, tip, colour)

##-Crop the height to the content; the width stays FIGURE_WIDTH_CM-##
y_top = min(-pad, min(top for top, _ in box_edges) - CROP_MARGIN_PX)
y_bottom = max(image_height + pad, max(bottom for _, bottom in box_edges) + CROP_MARGIN_PX)
axis.set_ylim(y_bottom, y_top)
figure.set_size_inches(FIGURE_WIDTH_CM / 2.54, (y_bottom - y_top) * inches_per_unit)

##-Save at exactly FIGURE_WIDTH_CM wide (bbox_inches='tight' would change the width)-##
figure.patch.set_facecolor('white')
axis.set_facecolor('white')
for output_file in OUTPUT_FILES:
    figure.savefig(output_file, dpi=SAVE_DPI, facecolor='white', transparent=False)

plt.show()

The displayed figure is generated in memory. The notebook does not automatically save the figure or any intermediate rendering files.